# CogVideoX-2B Video Generation API — Kaggle
Runs THUDM/CogVideoX-2B on free T4/P100 GPU (30 hrs/week).
Exposes a Gradio API endpoint — set it as `T2V_API_URL` in your `.env` file.

## Setup
1. **Internet**: On → Settings ⚙ > Internet > Check "Enable internet"
2. **GPU**: On → Settings ⚙ > Accelerator > GPU T4 x1 or P100
3. **Run all** → Copy the Gradio link → Paste in `.env` as `T2V_API_URL`

In [ ]:
import torch, psutil, os, gc
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Settings > Accelerator > GPU T4 x1 or P100")
free, total = psutil.virtual_memory()[1:3]
print(f"System RAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
print("GPU check passed!")

In [ ]:
!pip install -q gradio diffusers transformers accelerate sentencepiece protobuf imageio[ffmpeg]
print("Dependencies installed!")

In [ ]:
import torch, gradio as gr, os, time, random
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

MODEL = "THUDM/CogVideoX-2b"
print(f"Loading {MODEL} (~6GB VRAM - fits T4/P100)...")

pipe = CogVideoXPipeline.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
)
pipe.to("cuda")
pipe.enable_model_cpu_offload()
print("Model loaded on GPU!")

In [ ]:
def generate(prompt, steps=25, guidance=6.0, seed=-1):
    if seed < 0:
        seed = random.randint(0, 2**31)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    print(f"Generating: {prompt}")
    t0 = time.time()
    result = pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
        num_frames=49
    ).frames[0]
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.1f}s")
    path = export_to_video(result, fps=8)
    return path

iface = gr.Interface(
    fn=generate,
    inputs=[
        gr.Textbox(label="Prompt", value="A cute red panda riding a bicycle through a futuristic city, cinematic"),
        gr.Slider(5, 50, 25, label="Inference Steps"),
        gr.Slider(1.0, 10.0, 6.0, label="Guidance Scale"),
        gr.Number(-1, label="Seed (-1=random)", precision=0),
    ],
    outputs=gr.Video(label="Generated Video"),
    title="CogVideoX-2B AI Video Generator",
    description="Free AI video generation on Kaggle T4/P100 GPU",
)

print("=" * 60)
print("COPY THE GRADIO LINK BELOW (xxx.gradio.app)")
print("Set it as T2V_API_URL in your .env file")
print("=" * 60)
iface.launch(share=True)